# ML-04 — Data Contract: Refresh / Content Opportunity Ranking

This notebook uses the March 2026 warehouse partition. It develops a decision-support ranking for which content pages deserve refresh review first.

## 1. Unit of analysis + time window

1. **One warehouse row** is one pseudonymized content page for one pseudonymized client on one report date: a page-day. For my lane, I roll page-days into one page-level feature row.
2. **Tables:** I use `fact_content_daily_performance` for daily GSC/GA4 activity. `dim_clients` and `dim_content` are context tables for a later history/metadata check, not model features here.
3. **Time window:** March 2026 (`month=2026-03`), a mid-panel month. Days 1–20 are the decision-time feature window; days 21–31 are the forward outcome window. June, including `_sample`, remains sealed.
4. **What I rank:** pages for refresh review. The proxy label is an impressions decline: forward daily-average impressions are below 80% of prior-window daily-average impressions.
5. **Deliberately excluded:** all days 21–31 activity from the honest feature set, because it happens after the decision moment and is label information.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

# Uses a local HF cache or the user's authenticated credential; no token is written to this notebook.
march_file = hf_hub_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet',
)
con = duckdb.connect()
daily = f"read_parquet('{march_file}')"
MONTH = "2026-03"
print(f'Loaded {MONTH} from the authenticated warehouse cache.')

Loaded 2026-03 from the authenticated warehouse cache.


## 2. Fields: feature / label / context / excluded

- **Features:** `imp_prior`, `clicks_prior`, `position_prior`, `active_days_prior`, `sessions_prior`.
- **Label / proxy:** `is_refresh_decline`, calculated from days 21–31 `gsc_impressions`.
- **Context:** `client_hash_id`, `content_hash_id`, and `report_date`, used only to group and time-split rows.
- **Excluded:** forward-window impressions/clicks, `gsc_data_available IS FALSE`, GA4 rows where `ga4_data_available IS FALSE`, and `gsc_avg_position = 0`. These are either unavailable, zero-filled rather than observed, or no-position data.

## 3. Verify it with exactly three queries

The three checks below test the stated page-day grain, March slice scale/date span, and GA4 availability. The availability check intentionally uses `IS TRUE`, not a zero-value test.

In [2]:
# Verification query 1 — grain: zero duplicate groups means page-day is unique.
q1 = con.sql(f"""
    SELECT COUNT(*) AS duplicate_page_day_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM {daily}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()
q1

,duplicate_page_day_groups
0,0


In [3]:
# Verification query 2 — March row count and observed date span.
q2 = con.sql(f"""
    SELECT COUNT(*) AS page_days,
           COUNT(DISTINCT content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id) AS clients,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM {daily}
""").df()
q2

,page_days,pages,clients,first_day,last_day
0,9841378,331437,55,2026-03-01,2026-03-31


In [4]:
# Verification query 3 — availability: count only rows that survive IS TRUE.
q3 = con.sql(f"""
    SELECT COUNT(*) AS ga4_available_page_days
    FROM {daily}
    WHERE ga4_data_available IS TRUE
""").df()
q3

,ga4_available_page_days
0,413966


### Five features — available when?

- **`imp_prior`** — knowable at the decision moment because it sums GSC impressions from days 1–20 only.
- **`clicks_prior`** — knowable at the decision moment because it sums GSC clicks from days 1–20 only.
- **`position_prior`** — knowable at the decision moment because it averages observed, non-zero position on prior days only.
- **`active_days_prior`** — knowable at the decision moment because it counts prior dates with at least one observed impression.
- **`sessions_prior`** — knowable at the decision moment because it sums GA4 sessions from days 1–20 only where `ga4_data_available IS TRUE`.

In [5]:
# Feature-frame query: one row per page, with an intentionally separate forward label window.
feature_frame = con.sql(f"""
    WITH observed_gsc AS (
        SELECT *, EXTRACT(day FROM report_date) AS day_of_month
        FROM {daily}
        WHERE gsc_data_available IS TRUE
    ), page_month AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(CASE WHEN day_of_month <= 20 THEN gsc_impressions ELSE 0 END) AS imp_prior,
            SUM(CASE WHEN day_of_month <= 20 THEN gsc_clicks ELSE 0 END) AS clicks_prior,
            AVG(CASE WHEN day_of_month <= 20 AND gsc_impressions > 0
                     THEN gsc_avg_position END) AS position_prior,
            COUNT(DISTINCT CASE WHEN day_of_month <= 20 AND gsc_impressions > 0
                                THEN report_date END) AS active_days_prior,
            SUM(CASE WHEN day_of_month <= 20 AND ga4_data_available IS TRUE
                     THEN ga4_sessions ELSE 0 END) AS sessions_prior,
            SUM(CASE WHEN day_of_month >= 21 THEN gsc_impressions ELSE 0 END) AS imp_forward
        FROM observed_gsc
        GROUP BY 1, 2
    )
    SELECT *
    FROM page_month
    WHERE imp_prior >= 100
""").df()

# The 20-day and 11-day windows are converted to daily averages before comparison.
feature_frame['is_refresh_decline'] = (
    feature_frame['imp_forward'] / 11 < 0.80 * feature_frame['imp_prior'] / 20
).astype(int)

print(f"Candidate pages: {len(feature_frame):,}; decline base rate: {feature_frame['is_refresh_decline'].mean():.3f}")
feature_frame[[
    'imp_prior', 'clicks_prior', 'position_prior', 'active_days_prior',
    'sessions_prior', 'is_refresh_decline'
]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate pages: 87,267; decline base rate: 0.318


,imp_prior,clicks_prior,position_prior,active_days_prior,sessions_prior,is_refresh_decline
0,646.0,2.0,12.414119,20,2.0,1
1,149.0,0.0,6.913376,19,0.0,1
2,240.0,0.0,8.452744,20,2.0,1
3,8151.0,15.0,2.498021,20,0.0,0
4,3836.0,47.0,3.329946,15,0.0,0


### The trap: a deliberately leaked label-derived column

`imp_forward` is the days 21–31 outcome used to create the label. I add it once to demonstrate leakage, record the inflated result, then remove it. The honest model retains only the five prior-window features.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

honest_features = [
    'imp_prior', 'clicks_prior', 'position_prior', 'active_days_prior', 'sessions_prior'
]
target = feature_frame['is_refresh_decline'].to_numpy()
groups = feature_frame['client_hash_id'].to_numpy()
train_idx, test_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
    .split(feature_frame, target, groups)
)

def score_auc(columns):
    model = RandomForestClassifier(
        n_estimators=120, random_state=42, n_jobs=-1, class_weight='balanced'
    )
    model.fit(feature_frame.iloc[train_idx][columns].fillna(0), target[train_idx])
    probabilities = model.predict_proba(feature_frame.iloc[test_idx][columns].fillna(0))[:, 1]
    return roc_auc_score(target[test_idx], probabilities)

honest_auc = score_auc(honest_features)
leaked_auc = score_auc(honest_features + ['imp_forward'])  # deliberate leak: outcome-window column

print(f"Honest five-feature ROC AUC: {honest_auc:.3f}")
print(f"With deliberate imp_forward leak: {leaked_auc:.3f}")
print("Leak column deleted from the feature list; the honest number is the one kept.")

Honest five-feature ROC AUC: 0.614
With deliberate imp_forward leak: 1.000
Leak column deleted from the feature list; the honest number is the one kept.


## 4. Data limitation

This is an unbalanced panel: clients began GSC and GA4 tracking at different times. In particular, GA4 zeros outside `ga4_data_available IS TRUE` are zero-filled missing measurement, not evidence of no engagement. This March-only, observational slice can prioritize review; it cannot establish that refreshing a page causes a ranking or traffic change.

## 5. Self-check

- [x] Five plain-language contract answers are present.
- [x] Exactly three verification queries have visible outputs.
- [x] The availability verification uses `IS TRUE`.
- [x] Five decision-time features and their availability statements are present.
- [x] The leak experiment is shown, then the leak column is removed.
- [x] No client names, URLs, private queries, or token appear in the notebook.